# 2 — Preprocessing: Data Cleaning, Datasets, and DataLoaders

This notebook prepares the GTZAN dataset for model training. This notebook

- Connects to PostgreSQL to pull authoritative train/val/test split assignments, and verifies data quality (nulls, duplicates, corrupt-file exclusion)
- Imports the `AudioDataset` class, audio helpers, and augmentations from `code/common.py`, and builds the PyTorch DataLoaders

**Output:** three ready-to-train DataLoaders (`train_loader`, `val_loader`, `test_loader`) that yield normalized mel spectrogram tensors and integer genre labels.

## 1. Imports and Configuration

The preprocessing pipeline uses `librosa` for audio loading and mel spectrogram extraction, then returns PyTorch tensors that can be passed directly into model training.

**Configuration decisions from EDA (Sprint 1):**

- `CLIP_DURATION_SECONDS = 30` - GTZAN standard; majority of files are ~30.013 s. Files shorter than 30 s are zero-padded; longer ones are truncated to keep all spectrograms the same shape.
- `SAMPLE_RATE = 22_050` - EDA (Section 7) confirmed all 999 loadable files share this rate, no resampling required.
- `N_MELS = 128` - Standard for audio CNNs; captures enough frequency resolution without excessive computation.
- `N_FFT = 2048` / `HOP_LENGTH = 512` - Standard STFT parameters at 22,050 Hz - ~23 ms windows with ~50% overlap.
- `RANDOM_SEED = 42` - Fixed for reproducibility.

**Shared code:** `load_split`, `pad_or_truncate`, `audio_to_mel_spectrogram`, `augment_waveform`, `augment_mel_spectrogram`, `AudioDataset`, and `AudioCNN` used to be copy-pasted across this notebook, `3_model.ipynb`, and `4_predict.ipynb` — three copies to keep in sync by hand. They now live once in `code/common.py` and are imported below. This preserves each notebook's "restart kernel, run top to bottom" independence — `common.py` is a plain source file, not a notebook with its own kernel state, so importing it is no different from importing `librosa`. The only thing lost versus copy-paste is that you can no longer hand someone a single self-contained `.ipynb`; you now need `common.py` alongside it (which, since it's committed source, is always the case in this repo).

In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("NUMBA_CACHE_DIR", str(Path.cwd() / ".numba-cache"))
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib-cache"))

# code/common.py lives alongside this notebook; ensure it's importable on a
# clean kernel restart regardless of exactly how Jupyter set cwd.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from common import RANDOM_SEED, find_repo_root, load_split, AudioDataset

BATCH_SIZE = 16

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 2. Metadata DataFrame from Database

`audio_clips` stores one row per wav file with its label (via FK to `labels`) and train/val/test split. We query only usable files - `is_corrupted = FALSE` excludes `jazz.00054.wav` and `is_duplicate = FALSE` excludes any repeated files. File paths are stored relative to the repo root and resolved to absolute paths so librosa can load them.

In [2]:
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

repo_root = find_repo_root(Path.cwd())
load_dotenv(repo_root / ".env")

DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_HOST     = "localhost"
DB_PORT     = 5432
DB_NAME     = "audio_genre_classifier"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

try:
    with engine.connect() as con:
        db_name = con.execute(text("SELECT current_database()")).fetchone()[0]
    print(f"Connected to: {db_name}")
except Exception as e:
    print(f"Connection failed: {e}")


metadata_df = pd.read_sql(
    sql=text("""
        SELECT ac.file_path, l.label_name AS label, ac.split
        FROM   audio_clips ac
        JOIN   labels l ON l.label_id = ac.label_id
        WHERE  ac.is_corrupted = FALSE
          AND  ac.is_duplicate = FALSE
        ORDER  BY ac.file_path
    """),
    con=engine,
)
metadata_df["file_path"] = metadata_df["file_path"].apply(
    lambda p: str(repo_root / p)
)

metadata_df.head()

Connected to: audio_genre_classifier


,file_path,label,split
0,/Users/ac/Projects/tkh_projects/audio-genre-cl...,blues,val
1,/Users/ac/Projects/tkh_projects/audio-genre-cl...,blues,train
2,/Users/ac/Projects/tkh_projects/audio-genre-cl...,blues,train
3,/Users/ac/Projects/tkh_projects/audio-genre-cl...,blues,val
4,/Users/ac/Projects/tkh_projects/audio-genre-cl...,blues,train


In [3]:
print(metadata_df.shape)
print(metadata_df["split"].value_counts())
print(metadata_df["label"].value_counts().sort_index())

(999, 3)
split
train    699
val      150
test     150
Name: count, dtype: int64
label
blues        100
classical    100
country      100
disco        100
hiphop       100
jazz          99
metal        100
pop          100
reggae       100
rock         100
Name: count, dtype: int64


## 3. Data Cleaning

Checks run before any model training. Every decision is documented below the output.

1. **Null values** - the DB enforces `NOT NULL` on all columns, but we confirm here
2. **Duplicates** - each `file_path` is `UNIQUE` in the DB, so duplicates are impossible, but we verify the dataframe too
3. **Corrupt file exclusion** - `jazz.00054.wav` is in `audio_clips` with `is_corrupted = TRUE`. The query in Section 2 filters it out before it reaches this dataframe
4. **Class balance** - confirmed from EDA; verified here against the loaded splits
5. **Sample rate** - all files confirmed at 22,050 Hz in EDA; no resampling applied

In [4]:
null_count = metadata_df.isnull().sum().sum()
dup_count = metadata_df.duplicated().sum()
corrupt_count = metadata_df["file_path"].str.contains("jazz.00054", regex=False).sum()

print(f"Null values:          {null_count}  (expect 0)")
print(f"Duplicate rows:       {dup_count}  (expect 0)")
print(f"jazz.00054 rows:      {corrupt_count}  (expect 0 — filtered by is_corrupted = FALSE)")

# Verify the flag counts directly in the DB
with engine.connect() as con:
    flagged = con.execute(text("""
        SELECT
            SUM(is_corrupted::int) AS corrupted,
            SUM(is_duplicate::int) AS duplicates
        FROM audio_clips
    """)).fetchone()
print(f"\nIn DB — corrupted: {flagged[0]}  duplicates: {flagged[1]}")
print(f"Total songs in metadata_df: {len(metadata_df)}")

Null values:          0  (expect 0)
Duplicate rows:       0  (expect 0)
jazz.00054 rows:      0  (expect 0 — filtered by is_corrupted = FALSE)

In DB — corrupted: 1  duplicates: 0
Total songs in metadata_df: 999


**Cleaning decisions:**

- Corrupt file 
    - `jazz.00054.wav` fails to load entirely (EDA Section 8). It is stored in `audio_clips` with `is_corrupted = TRUE` and excluded from all training queries. It is kept in the database rather than deleted to maintain an audit record of what was removed and why.
- Duplicates 
    - GTZAN contains no duplicate files. `is_duplicate = FALSE` for all 1,000 rows. No action needed.
- Class balance 
    - EDA confirmed exactly 100 songs per genre (jazz has 99 usable due to the corrupt file). The dataset is effectively balanced — no class weighting or oversampling is applied.
- Sample rate 
    - EDA (Section 7) confirmed all 999 loadable files are 22,050 Hz. No resampling is applied.
- Clip length 
    - 10 songs are marginally short and produce only 9 three-second clips in `features_3_sec.csv`. For the raw audio pipeline used here, these files are zero-padded to `FIXED_NUM_SAMPLES` (660,150 samples = 30 s at 22,050 Hz) by `pad_or_truncate()`. No files are dropped.

## 4. Label Encoding

PyTorch classification targets should be integer class IDs. The mapping is built from the dataframe labels so the same encoding can be reused during training and prediction.

In [5]:
labels = sorted(metadata_df["label"].unique())
label_to_idx = {label: idx for idx, label in enumerate(labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

label_to_idx

{'blues': 0,
 'classical': 1,
 'country': 2,
 'disco': 3,
 'hiphop': 4,
 'jazz': 5,
 'metal': 6,
 'pop': 7,
 'reggae': 8,
 'rock': 9}

## 5. Audio Helpers

`pad_or_truncate`, `audio_to_mel_spectrogram`, `augment_waveform`, and `augment_mel_spectrogram` are imported from `common.py` (shared with `3_model.ipynb` and `4_predict.ipynb` so all three notebooks use the exact same preprocessing logic). Each clip is loaded at the same sample rate, then padded or truncated to a fixed number of samples. This keeps every mel spectrogram the same shape, which lets the DataLoader stack examples into batches.

## 6. PyTorch Dataset

`AudioDataset` is imported from `common.py` (shared with `3_model.ipynb` and `4_predict.ipynb`). It accepts a dataframe with `file_path`, `label`, and `split`, validating those columns are present. Augmentation is only enabled for rows where `split == 'train'` and `augment=True`, so validation and test examples remain stable.

## 7. Train, Validation, and Test DataLoaders

The training loader shuffles examples and uses augmentation. Validation and test loaders do not shuffle or augment so metrics are repeatable.

In [6]:
train_df = load_split("train", engine, repo_root)
val_df   = load_split("val",   engine, repo_root)
test_df  = load_split("test",  engine, repo_root)

train_dataset = AudioDataset(train_df, label_to_idx=label_to_idx, augment=True)
val_dataset = AudioDataset(val_df, label_to_idx=label_to_idx, augment=False)
test_dataset = AudioDataset(test_df, label_to_idx=label_to_idx, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train examples: {len(train_dataset)}")
print(f"Val examples  : {len(val_dataset)}")
print(f"Test examples : {len(test_dataset)}")

Train examples: 699
Val examples  : 150
Test examples : 150


## 8. Verification Cell

Confirming that the DataLoader can produce a batch, that features and labels have the expected tensor types, and that spectrogram values are finite.

Training on all 999 files at once would require loading gigabytes of audio into memory. Instead, PyTorch uses two layers:

- **`Dataset`** - knows how to load one file: takes a file path, loads the audio, converts it to a mel spectrogram tensor, and returns `(spectrogram, label)`
- **`DataLoader`** - wraps the Dataset and handles batching (groups 16 examples together), shuffling (randomizes order each epoch), and background loading (loads the next batch while the CNN trains on the current one)

This is the expected correct shape of tensor, the DataLoader is the standard PyTorch way to produce it. It won't load any audio until you ask it for a batch. 

This cell catches that before a training run that could take hours. One batch loaded successfully means the whole pipeline works end to end. Checking before training helps with ensuring that we catch set up failures before we start training which could take hours.

**Expected output shape:** `[16, 1, 128, 1292]`

- 16 - batch size, 16 songs per training step
- 1 - channels, mono audio treated as a single-channel image
- 128 - mel frequency bins, 128 frequency bands from low to high
- 1292 - time frames, 30 s × 22,050 samples/sec ÷ 512 (HOP_LENGTH) = 1,292 steps

In [7]:
batch_features, batch_labels = next(iter(train_loader))

print(f"Batch feature shape : {batch_features.shape}")
print(f"Batch feature dtype  : {batch_features.dtype}")
print(f"Batch label shape   : {batch_labels.shape}")
print(f"Batch label dtype    : {batch_labels.dtype}")
print(f"Feature min value   : {batch_features.min().item():.4f}")
print(f"Feature max value   : {batch_features.max().item():.4f}")
print(f"All feature values finite: {torch.isfinite(batch_features).all().item()}")

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Batch feature shape : torch.Size([16, 1, 128, 1292])
Batch feature dtype  : torch.float32
Batch label shape   : torch.Size([16])
Batch label dtype    : torch.int64
Feature min value   : -2.9484
Feature max value   : 4.5103
All feature values finite: True


## Summary

This notebook takes raw GTZAN audio files from `data/genres_original/` and produces three PyTorch DataLoaders ready for CNN training.

**What was done:**
- Connected to PostgreSQL and pulled train/val/test split assignments from `audio_clips` JOIN `labels`, filtered by `is_corrupted = FALSE` and `is_duplicate = FALSE`
- Verified data quality: zero nulls, zero duplicates, corrupt file confirmed absent from training data
- Mel spectrogram pipeline (imported from `code/common.py`): load WAV → pad/truncate to 30 s → log-mel spectrogram → per-clip z-score normalization → PyTorch tensor
- Applied augmentation on the training split only (waveform: gain, time shift, noise; spectrogram: frequency masking)
- Wrapped each split in `AudioDataset` (from `code/common.py`) and built three `DataLoader` instances

**Decisions made:**

| Decision | Choice | Reason |
|---|---|---|
| Clip length | 30 seconds | GTZAN standard; majority of files are ~30.013 s (EDA Section 6) |
| Short clips | Zero-pad to 30 s | 10 songs are marginally short; padding avoids dropping data |
| Sample rate | 22,050 Hz | All GTZAN files share this rate — no resampling needed (EDA Section 7) |
| Corrupt file | Flag in DB (`is_corrupted = TRUE`) | `jazz.00054.wav` cannot load; kept in DB as audit record, filtered from queries |
| Duplicates | No action | GTZAN has no duplicate files |
| Class imbalance | No action | Dataset is effectively balanced — 100 songs per genre (99 for jazz) |
| Spectrogram | Log-mel, 128 bins | Standard for audio CNNs; perceptually meaningful frequency representation |
| Normalization | Per-clip z-score | Removes per-file amplitude variation; stabilizes model input range |
| Augmentation | Training split only | Val and test sets unchanged for consistent, repeatable evaluation |
| Split source | PostgreSQL `audio_clips` | Reproducible across all teammates via fixed random seed (42) |

**Output batch shape:** `[16, 1, 128, 1292]` — (batch, channels, mel bins, time frames)